# Lesson 5a: Convolutional Networks — Theory

Every network so far has flattened its input into a single vector before
the first `nn.Linear`/`W @ x`, discarding the fact that pixels are
arranged on a 2D grid. A digit shifted one pixel to the right becomes a
completely different input vector to a flattening MLP, which must
re-learn every feature at every location independently — 3072 pixels in
and 128 hidden units out means 393,216 independent weights before the
network has learned anything about *images* specifically. Convolution
fixes this by building two assumptions about images directly into the
layer: **local connectivity** (an output depends on a small neighbourhood
of the input, not the whole image) and **parameter sharing** (the same
small set of weights is reused at every location). This notebook derives
both, the arithmetic that governs a convolutional layer's output
size and receptive field, and pooling — then implements a 2D convolution's
forward and backward pass from scratch in NumPy, verified against
PyTorch.

By the end of this notebook you will have:
- derived **convolution as a constrained linear operator** and quantified
  the parameter-count reduction against a dense layer on the same input,
- derived the **output-size formula** governing padding, stride and
  dilation, verified against PyTorch for several parameter combinations,
- derived the **receptive field** of a stack of convolutional layers and
  confirmed it empirically via autograd,
- implemented **max pooling from scratch** and measured the local
  translation-invariance it buys, and
- implemented a **2D convolution's forward and backward pass from
  scratch in NumPy**, verified numerically against `torch.nn.functional.conv2d`
  and its autograd gradients.

## Introduction

3a-4b's networks all took a flattened pixel vector as input — a valid way
to feed an image to a network, but one that throws away everything the
network could otherwise exploit about images specifically: nearby pixels
are related, and a useful feature (an edge, a corner, a texture) is
useful wherever in the image it appears. A **convolutional layer** encodes
both facts directly into the layer's structure, rather than hoping an MLP
learns them from data: each output unit looks at only a small local patch
of the input (**local connectivity**), and the same small set of weights —
a **kernel** — produces every output unit, slid across every position in
the image (**parameter sharing**, also called **translation
equivariance**: shift the input, and the output shifts the same way,
rather than being computed by an entirely different set of weights). The
rest of this notebook derives exactly what that buys, in parameters, in
receptive field, and in the invariances pooling adds on top — then builds
the operation from scratch to show none of it is architecture magic, just
matrix multiplication done a specific, sparse way.

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, data
# subsampling) is reproducible.
import io
import pathlib
import urllib.request

import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt
import pandas as pd
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from PIL import Image

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

### Loading MNIST and CIFAR-10

MNIST via `torchvision.datasets.MNIST` (a reliable host, as in 2a/2b);
CIFAR-10 via the Hugging Face parquet mirror (3a/4a's workaround for the
unreliably slow canonical torchvision host). Only a handful of images are
needed here — this notebook demonstrates operators and their arithmetic,
not a trained model.

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
mnist = torchvision.datasets.MNIST(root="data", train=True, download=True, transform=transform)
mnist_image = mnist[0][0].unsqueeze(0)  # (1, 1, 28, 28)

CIFAR_BASE = "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text"


def load_cifar10_subset(split, n, seed):
    path = pathlib.Path("data") / f"cifar10_{split}.parquet"
    path.parent.mkdir(exist_ok=True)
    if not path.exists():
        urllib.request.urlretrieve(f"{CIFAR_BASE}/{split}-00000-of-00001.parquet", path)
    df = pd.read_parquet(path)
    g = np.random.default_rng(seed)
    idx = g.permutation(len(df))[:n]
    images = np.stack([
        np.asarray(Image.open(io.BytesIO(df.iloc[i]["img"]["bytes"])), dtype=np.float32) / 255.0
        for i in idx
    ])
    return images


cifar_images = load_cifar10_subset("train", 4, seed=SEED)
cifar_image = torch.tensor(cifar_images[0]).permute(2, 0, 1).unsqueeze(0)  # (1, 3, 32, 32)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(mnist_image[0, 0], cmap="gray"); axes[0].set_title("MNIST (28x28x1)"); axes[0].axis("off")
axes[1].imshow(cifar_image[0].permute(1, 2, 0)); axes[1].set_title("CIFAR-10 (32x32x3)"); axes[1].axis("off")
plt.show()

## Convolution as a Linear Operator

2D (cross-correlation, what every deep learning framework calls
"convolution") slides a small kernel $K\in\mathbb R^{k\times k}$ over an
input $I$, producing at every position the inner product of the kernel
with the patch it currently covers:

$$(I * K)[i,j] = \sum_{u=0}^{k-1}\sum_{v=0}^{k-1} I[i+u,\,j+v]\; K[u,v].$$

This is **linear in $I$** for a fixed kernel — a sum of scaled input
values — so it can be written as an ordinary matrix-vector product $y =
Wx$, exactly like every `nn.Linear` in this series, for some matrix $W$
built from $K$. What makes $W$ a *convolution's* matrix rather than an
arbitrary dense one is two structural constraints: **every row of $W$ has
only $k^2$ nonzero entries** (each output depends on a $k\times k$
patch, not the whole image — local connectivity), and **every row is the
same $k^2$ values in a shifted position** (the same kernel produces every
output — parameter sharing). Removing either constraint recovers an
ordinary dense layer; convolution is what a linear layer looks like once
both are imposed on purpose.

The parameter count consequence is dramatic, and easy to state exactly:
a dense layer connecting two same-size feature maps has one weight per
(input pixel, output pixel) pair, while a convolutional layer has exactly
$k^2$ weights (per input/output channel pair), independent of image
size.

In [ ]:
def dense_param_count(in_h, in_w, out_h, out_w, in_c=1, out_c=1):
    return in_h * in_w * in_c * out_h * out_w * out_c + out_c


def conv_param_count(kernel_size, in_c=1, out_c=1):
    return kernel_size * kernel_size * in_c * out_c + out_c


in_h, in_w = 28, 28
out_h, out_w = 26, 26  # a 3x3 kernel, no padding, stride 1 on a 28x28 input
k = 3

dense_params = dense_param_count(in_h, in_w, out_h, out_w)
conv_params = conv_param_count(k)
print(f"dense layer, {in_h}x{in_w} -> {out_h}x{out_w}: {dense_params:,} parameters")
print(f"conv layer,  {k}x{k} kernel, same output size:  {conv_params:,} parameters")
print(f"reduction factor: {dense_params / conv_params:,.0f}x")
assert dense_params / conv_params > 10_000, "the parameter reduction should be dramatic"

A dense layer connecting two $28\times28$-scale feature maps needs over
half a million weights; a $3\times3$ convolutional kernel producing the
same-size output needs ten. The dense layer could, in principle, learn
anything the conv layer can and more — but it would need vastly more data
to do it, because it has no built-in reason to treat "detect this edge at
position $(5,5)$" and "detect this edge at position $(6,6)$" as the same
problem. The convolutional kernel treats them as literally the same
computation by construction.

## Padding, Stride and Output Size

Three parameters govern a convolution's output size beyond the kernel
size $k$ itself:

- **Padding** ($p$): zeros added around the input border before sliding
  the kernel. Without it, every convolution shrinks the spatial size (the
  kernel cannot center itself on a border pixel without running off the
  edge); "same" padding chooses $p$ so the output size matches the input.
- **Stride** ($s$): how many pixels the kernel moves between applications.
  $s=1$ computes an output at every position; $s>1$ skips positions,
  downsampling the output — a cheap alternative to a separate pooling
  layer.
- **Dilation** ($d$): spacing inserted *between* the kernel's own taps, so
  a $3\times3$ kernel with $d=2$ samples a $5\times5$ area of the input
  while still only learning 9 weights — a way to grow the receptive field
  (next section) without adding parameters or downsampling.

Combining all three, a dilated kernel effectively spans $d(k-1)+1$ input
positions, so the output size along one dimension is

$$\text{out} = \left\lfloor \frac{\text{in} + 2p - d(k-1) - 1}{s} \right\rfloor + 1.$$

In [ ]:
def conv_output_size(in_size, kernel_size, stride=1, padding=0, dilation=1):
    return (in_size + 2 * padding - dilation * (kernel_size - 1) - 1) // stride + 1


configs = [
    dict(in_size=28, kernel_size=3, stride=1, padding=0, dilation=1),
    dict(in_size=28, kernel_size=3, stride=1, padding=1, dilation=1),  # "same" padding
    dict(in_size=28, kernel_size=3, stride=2, padding=1, dilation=1),
    dict(in_size=32, kernel_size=5, stride=1, padding=2, dilation=1),
    dict(in_size=32, kernel_size=3, stride=1, padding=0, dilation=2),  # dilated
]

for cfg in configs:
    predicted = conv_output_size(**cfg)
    x = torch.randn(1, 1, cfg["in_size"], cfg["in_size"])
    w = torch.randn(1, 1, cfg["kernel_size"], cfg["kernel_size"])
    actual = F.conv2d(x, w, stride=cfg["stride"], padding=cfg["padding"], dilation=cfg["dilation"]).shape[-1]
    status = "match" if predicted == actual else "MISMATCH"
    print(f"{cfg} -> predicted {predicted}, torch gives {actual}  [{status}]")
    assert predicted == actual

Every configuration's formula-predicted size matches PyTorch's actual
output exactly, including the dilated case — dilation grows the kernel's
effective footprint ($d(k-1)+1 = 5$ for a $3\times3$ kernel at $d=2$)
without changing the number of learned weights at all.

## Receptive Fields

A single convolutional layer's output depends on a $k\times k$ patch of
its input. Stack a second layer on top, and one of *its* outputs depends
on a $k\times k$ patch of the *first* layer's output — which itself
depends on a $k\times k$ patch of the original input for each of those
positions. The **receptive field** is the size of that accumulated
region in the original input. For stride-1, dilation-1 layers each with
kernel size $k$, each additional layer grows the receptive field by
exactly $k-1$:

$$R_L = 1 + \sum_{l=1}^{L}(k_l - 1).$$

With non-unit strides the growth compounds by however much the input has
already been downsampled by previous layers — a layer's kernel now spans
$k_l$ positions *of an already-strided grid*, so its contribution to the
receptive field is scaled by the cumulative stride ("jump") of everything
before it:

$$R_l = R_{l-1} + (k_l - 1)\,d_l \cdot \text{jump}_{l-1}, \qquad
\text{jump}_l = \text{jump}_{l-1}\cdot s_l.$$

Two stacked $3\times3$ convolutions reach a $5\times5$ receptive field
($1 + 2\times2 = 5$) using $2\times9=18$ weights per channel pair and an
extra non-linearity in between — the reasoning behind VGG-style networks
preferring several small kernels stacked deep over one large kernel.

In [ ]:
def receptive_field(layers):
    """layers: list of (kernel_size, stride, dilation)."""
    r, jump = 1, 1
    for k, s, d in layers:
        r += (k - 1) * d * jump
        jump *= s
    return r


def empirical_receptive_field(layers, canvas_size=41):
    x = torch.zeros(1, 1, canvas_size, canvas_size, requires_grad=True)
    h = x
    for k, s, d in layers:
        w = torch.ones(1, h.shape[1], k, k)
        h = F.conv2d(h, w, stride=s, dilation=d)
    center = tuple(dim // 2 for dim in h.shape[-2:])
    h[0, 0, center[0], center[1]].backward()
    rows_touched = torch.nonzero(torch.any(x.grad[0, 0] != 0, dim=1)).flatten()
    return int(rows_touched.max() - rows_touched.min() + 1)


layer_configs = [
    [(3, 1, 1), (3, 1, 1)],
    [(3, 1, 1), (3, 1, 1), (3, 1, 1)],
    [(3, 1, 1), (3, 2, 1), (3, 1, 1)],
    [(3, 1, 2)],  # single dilated layer
]

for layers in layer_configs:
    predicted = receptive_field(layers)
    empirical = empirical_receptive_field(layers)
    print(f"layers={layers} -> predicted RF={predicted}, empirical RF={empirical}  "
          f"[{'match' if predicted == empirical else 'MISMATCH'}]")
    assert predicted == empirical

The empirical check backs a single output unit's gradient all the way
to the input and measures how wide a band of input pixels received a
nonzero gradient — the formula predicts that width exactly, in every
configuration tested, including the strided and dilated cases where the
"one extra layer, $k-1$ more pixels" intuition alone would give the wrong
answer.

## Pooling

**Pooling** downsamples a feature map the way a strided convolution
does, but with no learned weights: a max-pooling layer slides a
$k\times k$ window across the input and keeps only the maximum value in
each window; average pooling keeps the mean instead. Pooling serves two
purposes convolution alone does not: it reduces spatial resolution
cheaply (fewer positions for every later layer to process), and it buys a
small amount of **local translation invariance** — if the strongest
activation in a window shifts by a pixel or two but stays inside the same
window, the max-pooled output is completely unchanged, whereas the raw
(unpooled) feature map shifts along with it.

In [ ]:
def maxpool2d_forward(X, size=2, stride=2):
    N, C, H, W = X.shape
    H_out, W_out = (H - size) // stride + 1, (W - size) // stride + 1
    out = np.empty((N, C, H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            window = X[:, :, i * stride:i * stride + size, j * stride:j * stride + size]
            out[:, :, i, j] = window.max(axis=(2, 3))
    return out


# A fixed edge-detecting kernel (not learned -- just a concrete feature map to pool).
edge_kernel = torch.tensor([[[[-1., -1., -1.], [-1., 8., -1.], [-1., -1., -1.]]]])
image = mnist_image
image_shifted = torch.roll(image, shifts=1, dims=3)  # shift 1 pixel right

feature_map = F.conv2d(image, edge_kernel, padding=1).detach().numpy()
feature_map_shifted = F.conv2d(image_shifted, edge_kernel, padding=1).detach().numpy()

pooled = maxpool2d_forward(feature_map, size=2, stride=2)
pooled_shifted = maxpool2d_forward(feature_map_shifted, size=2, stride=2)


def fraction_changed(a, b, tol=1e-5):
    return (np.abs(a - b) > tol).mean()


unpooled_change = fraction_changed(feature_map, feature_map_shifted)
pooled_change = fraction_changed(pooled, pooled_shifted)
print(f"fraction of elements changed by a 1-pixel shift, unpooled feature map: {unpooled_change:.3f}")
print(f"fraction of elements changed by a 1-pixel shift, max-pooled feature map:  {pooled_change:.3f}")
assert pooled_change < unpooled_change, "pooling should reduce sensitivity to a small shift"

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(feature_map[0, 0], cmap="gray"); axes[0].set_title("unpooled edge map"); axes[0].axis("off")
axes[1].imshow(pooled[0, 0], cmap="gray"); axes[1].set_title("max-pooled (2x2)"); axes[1].axis("off")
plt.show()

Shifting the input by a single pixel changes a much smaller fraction of
the max-pooled feature map than of the raw convolution output: many
$2\times2$ pooling windows still contain the same maximum value after the
shift, even though every individual pixel inside those windows moved.
That robustness to small, local shifts — not larger spatial reach, which
convolution and stride already provide — is what pooling specifically
contributes.

## Convolution from Scratch

Every convolution above ran through `torch.nn.functional.conv2d`. The
"Convolution as a Linear Operator" section's argument — a convolution is
just a very structured matrix multiply — suggests a direct
implementation: extract every $k\times k$ patch the kernel would see
(**im2col**), flatten the kernel to a matrix, and multiply. This is
exactly how many production convolution implementations work internally,
not a simplification made for teaching purposes.

In [ ]:
def conv2d_forward(X, W, b, stride=1, padding=0):
    """X: (N, C_in, H, W). W: (C_out, C_in, kH, kW). b: (C_out,)."""
    if padding > 0:
        X = np.pad(X, ((0, 0), (0, 0), (padding, padding), (padding, padding)))
    N, C_in, Hp, Wp = X.shape
    C_out, _, kH, kW = W.shape
    H_out = (Hp - kH) // stride + 1
    W_out = (Wp - kW) // stride + 1

    # im2col: every (C_in, kH, kW) patch the kernel will see, for every output position.
    cols = np.empty((N, C_in, kH, kW, H_out, W_out))
    for i in range(kH):
        for j in range(kW):
            cols[:, :, i, j, :, :] = X[:, :, i:i + stride * H_out:stride, j:j + stride * W_out:stride]
    cols = cols.reshape(N, C_in * kH * kW, H_out * W_out)

    W_flat = W.reshape(C_out, -1)  # convolution IS this matrix multiply
    out = np.einsum("oc,ncp->nop", W_flat, cols) + b.reshape(1, -1, 1)
    out = out.reshape(N, C_out, H_out, W_out)
    return out, (X.shape, W, stride, padding, cols, H_out, W_out)


def conv2d_backward(dOut, cache):
    X_shape, W, stride, padding, cols, H_out, W_out = cache
    N, C_in, Hp, Wp = X_shape
    C_out, _, kH, kW = W.shape

    dOut_flat = dOut.reshape(N, C_out, -1)
    W_flat = W.reshape(C_out, -1)

    db = dOut_flat.sum(axis=(0, 2))
    dW = np.einsum("nop,ncp->oc", dOut_flat, cols).reshape(W.shape)

    dcols = np.einsum("oc,nop->ncp", W_flat, dOut_flat)
    dcols = dcols.reshape(N, C_in, kH, kW, H_out, W_out)
    dX_padded = np.zeros((N, C_in, Hp, Wp))
    for i in range(kH):
        for j in range(kW):
            dX_padded[:, :, i:i + stride * H_out:stride, j:j + stride * W_out:stride] += dcols[:, :, i, j, :, :]
    dX = dX_padded[:, :, padding:Hp - padding, padding:Wp - padding] if padding > 0 else dX_padded
    return dX, dW, db

Verification against PyTorch, both directions: the forward pass must
match `F.conv2d` numerically, and the backward pass must match the
gradients PyTorch's autograd computes for the identical operation.

In [ ]:
rng = np.random.default_rng(SEED)
N, C_in, C_out, H, Wd, k, stride, padding = 2, 3, 4, 10, 10, 3, 2, 1

X_np = rng.normal(size=(N, C_in, H, Wd))
W_np = rng.normal(size=(C_out, C_in, k, k)) * 0.1
b_np = rng.normal(size=(C_out,)) * 0.1

out_np, cache = conv2d_forward(X_np, W_np, b_np, stride=stride, padding=padding)

X_t = torch.tensor(X_np, requires_grad=True)
W_t = torch.tensor(W_np, requires_grad=True)
b_t = torch.tensor(b_np, requires_grad=True)
out_t = F.conv2d(X_t, W_t, b_t, stride=stride, padding=padding)

forward_diff = np.max(np.abs(out_np - out_t.detach().numpy()))
print(f"forward pass: output shape {out_np.shape}, max abs difference vs torch: {forward_diff:.2e}")
assert np.allclose(out_np, out_t.detach().numpy(), atol=1e-8)

dOut = rng.normal(size=out_np.shape)
dX_np, dW_np, db_np = conv2d_backward(dOut, cache)

out_t.backward(torch.tensor(dOut))
dX_diff = np.max(np.abs(dX_np - X_t.grad.numpy()))
dW_diff = np.max(np.abs(dW_np - W_t.grad.numpy()))
db_diff = np.max(np.abs(db_np - b_t.grad.numpy()))
print(f"backward pass: max abs difference -- dX: {dX_diff:.2e}, dW: {dW_diff:.2e}, db: {db_diff:.2e}")
assert np.allclose(dX_np, X_t.grad.numpy(), atol=1e-8)
assert np.allclose(dW_np, W_t.grad.numpy(), atol=1e-8)
assert np.allclose(db_np, b_t.grad.numpy(), atol=1e-8)
print("from-scratch conv2d forward and backward match torch to floating-point precision")

Both directions agree with PyTorch to floating-point precision — the
im2col-and-matmul implementation is not an approximation of what a
convolution does, it is a literal, from-scratch realisation of the
"structured linear operator" argument this notebook opened with. Applying
it to real images with a hand-designed kernel makes the operation
tangible before a network ever learns one:

In [ ]:
sobel_x = np.array([[[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]], dtype=np.float64).reshape(1, 1, 3, 3)
sobel_y = np.array([[[-1, -2, -1], [0, 0, 0], [1, 2, 1]]], dtype=np.float64).reshape(1, 1, 3, 3)
bias_zero = np.zeros(1)

mnist_np = mnist_image.numpy().astype(np.float64)
edges_x, _ = conv2d_forward(mnist_np, sobel_x, bias_zero, stride=1, padding=1)
edges_y, _ = conv2d_forward(mnist_np, sobel_y, bias_zero, stride=1, padding=1)
edge_magnitude = np.sqrt(edges_x ** 2 + edges_y ** 2)

cifar_np = cifar_image.mean(dim=1, keepdim=True).numpy().astype(np.float64)  # grayscale for a single-channel kernel
cifar_edges_x, _ = conv2d_forward(cifar_np, sobel_x, bias_zero, stride=1, padding=1)
cifar_edges_y, _ = conv2d_forward(cifar_np, sobel_y, bias_zero, stride=1, padding=1)
cifar_edge_magnitude = np.sqrt(cifar_edges_x ** 2 + cifar_edges_y ** 2)

fig, axes = plt.subplots(2, 2, figsize=(7, 7))
axes[0, 0].imshow(mnist_np[0, 0], cmap="gray"); axes[0, 0].set_title("MNIST"); axes[0, 0].axis("off")
axes[0, 1].imshow(edge_magnitude[0, 0], cmap="gray"); axes[0, 1].set_title("Sobel edges (from-scratch conv2d)"); axes[0, 1].axis("off")
axes[1, 0].imshow(cifar_image[0].permute(1, 2, 0)); axes[1, 0].set_title("CIFAR-10"); axes[1, 0].axis("off")
axes[1, 1].imshow(cifar_edge_magnitude[0, 0], cmap="gray"); axes[1, 1].set_title("Sobel edges (from-scratch conv2d)"); axes[1, 1].axis("off")
plt.tight_layout()
plt.show()

A hand-designed Sobel kernel run through our own `conv2d_forward` picks
out edges in both a handwritten digit and a real photograph, using
exactly the same six-line implementation verified against PyTorch above.
A trained convolutional network's first layer learns kernels like this
one directly from data — 5b reproduces this notebook's operators with
`nn.Conv2d` and trains a real network to find out what it learns.

## Key Takeaways

- **Convolution is a linear operator with two structural constraints**:
  local connectivity (each output depends on a small $k\times k$ patch)
  and parameter sharing (the same kernel produces every output). Removing
  either recovers an ordinary dense layer; a $3\times3$ kernel needs 10
  parameters where a same-size dense layer on a $28\times28$ image needs
  over half a million.
- **Padding, stride and dilation** are governed by one output-size
  formula, $\lfloor(\text{in}+2p-d(k-1)-1)/s\rfloor+1$, verified against
  PyTorch for ordinary, "same"-padded, strided and dilated configurations.
- **Receptive field** grows by $(k_l-1)\cdot d_l\cdot\text{jump}_{l-1}$ at
  each stacked layer, confirmed empirically by backpropagating a single
  output unit's gradient to the input and measuring how wide a band of
  input pixels received nonzero gradient.
- **Pooling** adds local translation invariance pure convolution and
  stride do not: a 1-pixel input shift changed a measurably smaller
  fraction of a max-pooled feature map than of the raw, unpooled one.
- **A from-scratch NumPy `conv2d_forward`/`conv2d_backward`, built as
  im2col plus matrix multiplication**, matched `torch.nn.functional.conv2d`
  and its autograd gradients to floating-point precision in both
  directions — confirming the "convolution is a structured linear
  operator" argument this notebook opened with is exact, not a loose
  analogy — and applying it with a hand-designed Sobel kernel extracted
  real edges from both an MNIST digit and a CIFAR-10 photograph.